# 01 — Ingestão e tratamento dos dados

**Entregável E1/E2** (IBM8924 — AC de Projeto, Grupo 1: classificação de qualidade de gramado por imagens de satélite).

Este notebook constrói o dataset de imagens rotuladas usado nos notebooks seguintes
(`02_caracterizacao.ipynb`, `03_baseline.ipynb`):

1. Amostra pontos de pastagem no Google Earth Engine, rotulados pelo produto
   **MapBiomas Pasture Vigor** (proxy de qualidade — vigor baixo/médio/alto), mascarados
   para pixels classificados como Pastagem pelo MapBiomas LULC.
2. Para cada ponto, extrai um recorte (patch) de imagem **Sentinel-2 L2A** (bandas brutas,
   não embeddings) de tamanho fixo.
3. Separa os dados em treino/validação/teste (70/15/15) garantindo que recortes
   geograficamente próximos não fiquem em partições diferentes.
4. Salva o dataset final em `data/processed/`.

**Fonte / período / licença:** MapBiomas Coleção 9 (`mapbiomas-public`, uso público com
atribuição, brasil.mapbiomas.org), anos de referência 2019–2022; Sentinel-2 L2A
harmonizado (`COPERNICUS/S2_SR_HARMONIZED`, ESA/Copernicus, acesso aberto), composições
sazonais (jun–set) dos mesmos anos. Ambos acessados via Google Earth Engine.

**Critério de rotulagem:** vigor MapBiomas (1=baixo, 2=médio, 3=alto), mascarado para
pixels de Pastagem (código de classe 15) — ver `TODO.md`, opção de rótulo proxy adotada.

**Nota:** este notebook faz chamadas de rede ao Earth Engine e pode levar dezenas de
minutos para rodar do início ao fim (a extração de patches é o passo mais demorado).
Preencha o tempo real de execução no README após rodar.

## 1. Setup

In [ ]:
# Preinstalado na maioria dos ambientes; -q deixa a saída silenciosa e é inofensivo
# rodar de novo mesmo se já instalado.
!pip install -q earthengine-api pandas numpy scikit-learn matplotlib


In [ ]:
import time

import ee
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)


## 2. Earth Engine — autenticação e inicialização

In [ ]:
# Mesmo projeto GCP usado em model/grassland_quality_finetuning.ipynb e em app.py.
EE_PROJECT_ID = 'projetograssquality'  # <-- ajuste se necessário

ee.Authenticate()
ee.Initialize(project=EE_PROJECT_ID)
print('Earth Engine inicializado.')


## 3. Área de estudo (AOI)

Mantemos a mesma caixa delimitadora usada no protótipo original em `model/`, perto de
Goiânia-GO — essa região foi escolhida porque existem dados de campo publicados e
validados (LAPIG) sobre degradação de pastagem ali, úteis para comparação futura
(ver seção de Trabalhos Futuros no README). Para chegar ao mínimo de 1500 recortes sem
expandir a área geograficamente, amostramos **múltiplos anos** do produto de vigor
(seção 5) em vez de múltiplas regiões.

In [ ]:
aoi = ee.Geometry.Rectangle([-49.6, -16.9, -48.9, -16.3])
print('AOI area (km^2):', aoi.area().divide(1e6).getInfo())


## 4. Assets do MapBiomas

Mesma lógica de carregamento/mascaramento usada em `model/grassland_quality_finetuning.ipynb`
(reaproveitada sem alterações) — reproduzida aqui para que este notebook rode de forma
independente.

In [ ]:
PASTURE_VIGOR_ASSET = 'projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_pasture_vigor_v1'
LULC_ASSET = 'projects/mapbiomas-public/assets/brazil/lulc/v1'

# MapBiomas código de classe para Pastagem — reconfirmar contra a legenda atual em
# brasil.mapbiomas.org antes de tirar conclusões dos resultados (pode mudar entre coleções).
PASTURE_CLASS_CODE = 15

pv_image = ee.Image(PASTURE_VIGOR_ASSET)
pv_band_names = pv_image.bandNames().getInfo()
print(f'Pasture Vigor: {len(pv_band_names)} bandas (uma por ano).')

lulc_col = ee.ImageCollection(LULC_ASSET)


def find_year_band(band_names, year):
    matches = [b for b in band_names if str(year) in b]
    return matches[0] if matches else band_names[0]


def build_masked_vigor_image(year):
    """Vigor MapBiomas do ano dado, mascarado para pixels de Pastagem (classe 15)."""
    vigor_band = find_year_band(pv_band_names, year)
    vigor_layer = pv_image.select(vigor_band)

    lulc_year_image = lulc_col.filter(ee.Filter.eq('year', year)).first()
    lulc_band = find_year_band(lulc_year_image.bandNames().getInfo(), year)

    pasture_mask = lulc_year_image.select(lulc_band).eq(PASTURE_CLASS_CODE)
    return vigor_layer.updateMask(pasture_mask).rename('vigor')


## 5. Amostragem estratificada multi-ano (≥1500 pontos)

O protótipo original amostrava só o ano de 2022 (100 pontos/classe = 300 no total),
abaixo do mínimo de 1500 exigido pelo enunciado. Para chegar lá **sem expandir a AOI**,
repetimos a amostragem estratificada em 4 anos do produto de vigor, tratando cada
(ano, classe) como uma amostragem independente e concatenando os resultados. O número de
pontos por (classe, ano) é definido com margem acima do necessário, pois alguns pontos
serão descartados depois por falta de dado de imagem (nuvem) na etapa 6.

In [ ]:
VIGOR_YEARS = [2019, 2020, 2021, 2022]
POINTS_PER_CLASS_PER_YEAR = 150  # 4 anos x 150 = 600/classe x 3 classes = 1800 pontos (margem sobre 1500)

all_rows = []
next_id = 0

for year in VIGOR_YEARS:
    vigor_masked = build_masked_vigor_image(year)
    sample_input = vigor_masked.addBands(ee.Image.pixelLonLat())

    stratified_points = sample_input.stratifiedSample(
        numPoints=POINTS_PER_CLASS_PER_YEAR,
        classBand='vigor',
        region=aoi,
        scale=30,
        seed=SEED,
        geometries=True,
    )
    points_info = stratified_points.getInfo()['features']
    print(f'Ano {year}: {len(points_info)} pontos amostrados')

    for f in points_info:
        coords = f['geometry']['coordinates']
        all_rows.append({
            'id': next_id,
            'lon': coords[0],
            'lat': coords[1],
            'label': f['properties']['vigor'],
            'vigor_year': year,
        })
        next_id += 1

points_df = pd.DataFrame(all_rows)

# Remoção de duplicatas: mesmo (lon, lat, vigor_year) não deveria ocorrer por
# construção do stratifiedSample, mas checamos explicitamente por segurança/rastreabilidade.
before = len(points_df)
points_df = points_df.drop_duplicates(subset=['lon', 'lat', 'vigor_year']).reset_index(drop=True)
print(f'Duplicatas removidas: {before - len(points_df)}')
print('Total de pontos amostrados:', len(points_df))
print(points_df.groupby(['vigor_year', 'label']).size())


## 6. Extração de patches Sentinel-2

Para cada ponto, extraímos um recorte fixo de `PATCH_SIZE x PATCH_SIZE` pixels
(64×64 a 10 m/pixel ⇒ ~640 m de lado) com as bandas `B2 (blue), B3 (green), B4 (red),
B8 (NIR), B11, B12 (SWIR)` do composto de mediana sazonal (jun–set, filtrado por
`CLOUDY_PIXEL_PERCENTAGE < 20`) do ano de vigor correspondente ao ponto.

**Tratamento de ausentes:** pixels sem observação válida no composto recebem
`defaultValue=0` (`sampleRectangle`); patches com fração alta de pixels zerados
(nuvem residual/sem cobertura) são descartados após a extração (ver checagem de
qualidade abaixo).

**Escala da extração:** chamadas síncronas em lotes pequenos (`BATCH_SIZE_EE` pontos
por vez) em vez de exportação em lote — mais simples de depurar, adequado para a ordem
de grandeza de pontos deste projeto (~1800 antes da checagem de qualidade).

In [ ]:
PATCH_SIZE = 64          # pixels
SCALE_M = 10             # metros/pixel (bandas Sentinel-2 usadas aqui são todas de 10m ou reamostradas)
HALF_SIDE_M = PATCH_SIZE / 2 * SCALE_M
BANDS_S2 = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
BATCH_SIZE_EE = 25       # pontos por rodada de getInfo(); mantém a resposta dentro do limite de request interativo
MAX_RETRIES = 3


def build_s2_mosaic(year):
    start, end = f'{year}-06-01', f'{year}-09-30'  # estação seca em GO -> menos nuvem
    col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .select(BANDS_S2)
    )
    return col.median()


_mosaic_cache = {}


def get_mosaic(year):
    if year not in _mosaic_cache:
        _mosaic_cache[year] = build_s2_mosaic(year)
    return _mosaic_cache[year]


def fetch_patch(row):
    mosaic = get_mosaic(row.vigor_year)
    region = ee.Geometry.Point([row.lon, row.lat]).buffer(HALF_SIDE_M).bounds()
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            patch = mosaic.sampleRectangle(region=region, defaultValue=0)
            info = patch.getInfo()['properties']
            arr = np.stack([np.array(info[b], dtype=np.float32) for b in BANDS_S2], axis=-1)
            # sampleRectangle pode variar +-1px perto das bordas; normaliza para o tamanho fixo.
            arr = arr[:PATCH_SIZE, :PATCH_SIZE, :]
            pad_h, pad_w = PATCH_SIZE - arr.shape[0], PATCH_SIZE - arr.shape[1]
            if pad_h > 0 or pad_w > 0:
                arr = np.pad(arr, ((0, max(pad_h, 0)), (0, max(pad_w, 0)), (0, 0)), mode='edge')
            return arr
        except Exception as e:
            if attempt == MAX_RETRIES:
                print(f'Falha ao extrair patch do ponto {row.id} (ano {row.vigor_year}): {e}')
                return None
            time.sleep(2 * attempt)
    return None


In [ ]:
patches = {}
rows = list(points_df.itertuples())

for start in range(0, len(rows), BATCH_SIZE_EE):
    batch = rows[start:start + BATCH_SIZE_EE]
    for row in batch:
        arr = fetch_patch(row)
        if arr is not None:
            patches[row.id] = arr
    done = min(start + BATCH_SIZE_EE, len(rows))
    print(f'{done}/{len(rows)} pontos processados ({len(patches)} com sucesso)')

print(f'\nExtração concluída: {len(patches)}/{len(points_df)} patches obtidos com sucesso.')


### Checagem de qualidade: descartar patches com muitos pixels sem dado

Pixels preenchidos por `defaultValue=0` indicam ausência de observação válida na
composição sazonal (nuvem persistente na janela jun–set, ou borda de cena). Patches
com mais de 5% dos pixels zerados em qualquer banda são descartados.

In [ ]:
MAX_MISSING_FRACTION = 0.05

valid_ids = []
for pid, arr in patches.items():
    missing_fraction = (arr == 0).mean()
    if missing_fraction <= MAX_MISSING_FRACTION:
        valid_ids.append(pid)

print(f'{len(valid_ids)}/{len(patches)} patches aprovados na checagem de qualidade '
      f'(<= {MAX_MISSING_FRACTION:.0%} de pixels sem dado).')

if len(valid_ids) < 1500:
    print('AVISO: total abaixo de 1500 apos a checagem de qualidade. '
          'Aumente POINTS_PER_CLASS_PER_YEAR ou adicione mais anos em VIGOR_YEARS e reexecute.')


In [ ]:
valid_df = points_df[points_df['id'].isin(valid_ids)].reset_index(drop=True)

X = np.stack([patches[pid] for pid in valid_df['id']]).astype(np.float32)
y_raw = valid_df['label'].to_numpy()

unique_labels = sorted(np.unique(y_raw))
label_map = {int(v): i for i, v in enumerate(unique_labels)}
y = np.array([label_map[int(v)] for v in y_raw], dtype=np.int64)

print('X shape (N, H, W, C):', X.shape)
print('Distribuição de classes:', dict(zip(*np.unique(y, return_counts=True))))
print('Mapeamento de rótulo (vigor MapBiomas -> classe do modelo):', label_map)


## 7. Split treino/val/teste (70/15/15) sem vazamento espacial

O enunciado exige que recortes da mesma cena não fiquem em partições diferentes.
Como os pontos são amostrados individualmente (não em grade regular), usamos
**blocking espacial**: cada ponto é atribuído a uma célula de grade de ~5,5 km
(`GRID_DEG = 0.05`, bem maior que o patch de 640 m), e **blocos inteiros** — não pontos
individuais — são alocados a uma única partição. Isso garante que dois pontos próximos
o bastante para compartilhar contexto de cena nunca caiam em partições diferentes.

In [ ]:
GRID_DEG = 0.05  # ~5.5 km por célula

valid_df = valid_df.copy()
valid_df['block_id'] = (
    (valid_df['lon'] / GRID_DEG).apply(np.floor).astype(int).astype(str)
    + '_'
    + (valid_df['lat'] / GRID_DEG).apply(np.floor).astype(int).astype(str)
)
valid_df['model_label'] = y

print(f"{valid_df['block_id'].nunique()} blocos únicos para {len(valid_df)} pontos.")


In [ ]:
def assign_blocks_to_splits(df, seed=SEED, target=(0.70, 0.15, 0.15)):
    """Aloca blocos inteiros (não pontos) a train/val/test.

    Algoritmo guloso: embaralha os blocos, e atribui cada um à partição que está
    mais abaixo da sua proporção-alvo no momento (em número de pontos).
    """
    rng = np.random.default_rng(seed)
    block_ids = df['block_id'].unique()
    rng.shuffle(block_ids)

    names = ['train', 'val', 'test']
    counts = {n: 0 for n in names}
    assignment = {}

    block_sizes = df.groupby('block_id').size().to_dict()

    for bid in block_ids:
        size = block_sizes[bid]
        total_so_far = sum(counts.values()) or 1
        deficits = {n: target[i] - counts[n] / total_so_far for i, n in enumerate(names)}
        chosen = max(deficits, key=deficits.get)
        assignment[bid] = chosen
        counts[chosen] += size

    return df['block_id'].map(assignment)


valid_df['split'] = assign_blocks_to_splits(valid_df)

# Verificação: nenhum bloco pode aparecer em mais de uma partição.
blocks_per_split = valid_df.groupby('block_id')['split'].nunique()
assert (blocks_per_split == 1).all(), 'Vazamento espacial detectado: bloco em mais de uma partição!'

print(valid_df.groupby('split').size())
print(valid_df.groupby(['split', 'model_label']).size().unstack(fill_value=0))


## 8. Salvar dataset processado (E1)

In [ ]:
import os

os.makedirs('../data/processed', exist_ok=True)

np.savez(
    '../data/processed/s2_patches.npz',
    X=X,
    y=y,
    split=valid_df['split'].to_numpy(),
    point_id=valid_df['id'].to_numpy(),
    lat=valid_df['lat'].to_numpy(),
    lon=valid_df['lon'].to_numpy(),
    vigor_year=valid_df['vigor_year'].to_numpy(),
    block_id=valid_df['block_id'].to_numpy(),
    band_names=np.array(BANDS_S2),
    label_map=label_map,
)

valid_df[['id', 'lat', 'lon', 'label', 'model_label', 'vigor_year', 'block_id', 'split']].to_csv(
    '../data/processed/points_metadata.csv', index=False
)

print('Salvo em data/processed/s2_patches.npz e data/processed/points_metadata.csv')
print('X:', X.shape, X.dtype, '| y:', y.shape, y.dtype)


## Resumo / rastreabilidade

- **Fontes:** MapBiomas Coleção 9 Pasture Vigor + LULC (rótulo/máscara), Sentinel-2 L2A
  harmonizado (imagem). Ambas públicas, acessadas via Google Earth Engine.
- **Período:** anos de vigor 2019–2022; composições Sentinel-2 sazonais (jun–set) dos
  mesmos anos.
- **Unidades:** bandas Sentinel-2 em refletância de superfície escalada por 10000
  (dividir por 10000 para refletância em [0,1] — feito em `dataset_utils.py`).
- **Ausentes:** pixels sem observação válida preenchidos com 0 (`defaultValue`);
  patches com >5% de pixels zerados descartados.
- **Duplicatas:** checadas e removidas por (lon, lat, vigor_year).
- **Rótulo:** vigor MapBiomas (1/2/3 = baixo/médio/alto), mascarado para pixels de
  Pastagem (código 15).
- **Split:** 70/15/15 por blocos espaciais de ~5,5 km, sem pontos da mesma vizinhança
  em partições diferentes; semente fixa (42).